# 第96课 · 白板见真章——合上电脑，亲手推导 FFT、CTC 和注意力机制

**学习目标**
1. 不看任何代码，能在白板上推导 FFT 蝶形公式
2. 不看任何代码，能写出 CTC 前向概率递推式
3. 不看任何代码，能推导 Scaled Dot-Product Attention 的矩阵形式
4. 发现自己的推导盲点，记录下来

> 这是整个课程里唯一一节**不允许运行代码格**的课。题型清单链回各课白板挑战（FFT / CTC / 注意力）。
> 每道题先在纸上推导，推完后才解锁"对答案"格。

← **上一课**　[L95 · 研究论文入门](L95_research_papers.ipynb)

> 上节课学习了 **研究论文入门**：三遍阅读法、论文结构写作、投稿流程与学术合作。  
> 本课将探讨 **白板演练**。

## 白板规则

- 纸 + 笔，不看 notebook，不看 src/
- 给自己计时：FFT ≤ 20 分钟，CTC ≤ 25 分钟，Attention ≤ 15 分钟
- 推导完再打开"对答案"格，不要提前偷看
- 记录：哪一步卡住了，卡了多久

## 🎯 别慌：这些山，你早就翻过了

今天的三道推导题**没有一个是新知识**——它们正是你之前亲手跨过的认知断崖：

- **FFT 蝶形**（L38-L39）：分治把 O(N²) 降到 O(N log N)
- **CTC 前向**（L68-L69）：用 blank 对齐变长序列，动态规划累加路径概率
- **Attention**（L83）：QKᵀ 打分 → softmax → 加权 V

白板演练考的不是"学没学过"，而是"能不能**合上电脑、脱稿复现**"。卡住很正常——卡在哪一步，就回哪一课重推一遍。

## ✏️ 题 1：FFT 蝶形推导（目标：20 分钟内完成）

**题目**

设信号长度 N = 8。

1. 写出 DFT 定义：X[k] = ?
2. 利用奇偶分解（odd-even decomposition），写出 Cooley-Tukey 分治（divide and conquer）递推式
3. 画出 N=8 的蝶形网络（8 个输入，3 层，每层 4 个蝶形单元（butterfly unit））
4. 写出一个蝶形单元的两条输出公式（用旋转因子（twiddle factor） W_N^k 表示）
5. 说明为什么时间复杂度从 O(N²) 降到 O(N log N)

先在纸上完成，再展开下面的"对答案"格。

In [ ]:
# 对答案格——先推导完再运行
import numpy as np

# DFT 定义
def naive_dft(x):
    N = len(x)
    return np.array([sum(x[n] * np.exp(-2j*np.pi*k*n/N) for n in range(N))
                     for k in range(N)])

# 蝶形单元：接受 E[k], O[k]，输出两个频率分量
def butterfly(E, O, k, N):
    twiddle = np.exp(-2j * np.pi * k / N)
    return E + twiddle * O, E - twiddle * O

# 验证：naive_dft vs np.fft.fft
x = np.random.randn(8)
assert np.allclose(naive_dft(x), np.fft.fft(x), atol=1e-9)
print('DFT 与 numpy 参考一致 ✅')
print()
print('蝶形单元示例（k=1, N=8）:')
E, O = np.exp(1j*0.3), np.exp(1j*0.7)
a, b = butterfly(E, O, k=1, N=8)
print(f'  输出 +: {a:.4f}')
print(f'  输出 -: {b:.4f}')

In [ ]:
# ✏️ 自评：在这里记录你的推导情况
fft_review = {
    "time_minutes":     None,   # 你用了多少分钟？
    "stuck_at":         None,   # 哪一步卡住了？
    "correct_parts":    None,   # 哪几步推对了？
    "need_practice":    None,   # 下次要加强哪里？
}
print({k: v for k, v in fft_review.items()})

## ✏️ 题 2：CTC 前向算法（目标：25 分钟内完成）

**题目**

设目标序列 Y = [a, b]，CTC 扩展后 π = [-, a, -, b, -]（- 是 blank）。

1. 写出 CTC 前向变量 α_t(s) 的定义
2. 写出初始条件 α_1(1) 和 α_1(2)
3. 写出递推公式（分三种情况：blank / 重复字符 / 普通转移）
4. 说明为什么 CTC 不需要对齐标注

先在纸上完成，再运行对答案格。

In [ ]:
# 对答案格——CTC 前向算法核心递推（简化版）
import numpy as np

def ctc_forward(log_probs, labels, blank=0):
    """
    log_probs: (T, V) 每帧每字符的 log 概率
    labels:    目标序列（不含 blank）
    返回：log P(Y | X)
    """
    T, V = log_probs.shape
    # 扩展标签：在每个字符前后插入 blank
    l_prime = [blank]
    for c in labels:
        l_prime += [c, blank]
    S = len(l_prime)

    NEG_INF = float('-inf')
    alpha = np.full((T, S), NEG_INF)

    # 初始化
    alpha[0, 0] = log_probs[0, blank]
    if S > 1:
        alpha[0, 1] = log_probs[0, l_prime[1]]

    # 递推
    for t in range(1, T):
        for s in range(S):
            c = l_prime[s]
            # 来自同位置
            a = alpha[t-1, s]
            # 来自前一位置
            if s > 0:
                a = np.logaddexp(a, alpha[t-1, s-1])
            # 跳过 blank（当前非 blank 且与前前不同）
            if s > 1 and c != blank and c != l_prime[s-2]:
                a = np.logaddexp(a, alpha[t-1, s-2])
            alpha[t, s] = a + log_probs[t, c]

    return np.logaddexp(alpha[T-1, S-1], alpha[T-1, S-2])

# 简单测试
T, V = 5, 3
np.random.seed(42)
raw = np.random.randn(T, V)
log_probs = raw - np.log(np.sum(np.exp(raw), axis=1, keepdims=True))
labels = [1, 2]  # 目标：字符 1 和字符 2
try:
    log_p = ctc_forward(log_probs, labels, blank=0)
    assert log_p is not None, "⬜ ctc_forward 未实现"
    assert np.isfinite(log_p), f"log 概率应为有限值，得到 {log_p}"
    assert log_p < 0, f"log 概率应为负数，得到 {log_p:.4f}"
    print(f'✅ CTC 前向验证通过，log P(Y|X) = {log_p:.4f}')
except (NotImplementedError, TypeError):
    print('⬜ ctc_forward 未实现')

In [ ]:
ctc_review = {
    "time_minutes":  None,
    "stuck_at":      None,
    "correct_parts": None,
    "need_practice": None,
}
print(ctc_review)

## ✏️ 题 3：Scaled Dot-Product Attention（目标：15 分钟内完成）

**题目**

1. 写出 Attention(Q, K, V) 的完整公式
2. 说明为什么要除以 √d_k（不是 d_k，也不是 1）
3. softmax 之后得到的矩阵形状是什么？
4. Multi-Head Attention 和单头的区别是什么（一句话）

先在纸上完成，再运行对答案格。

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Q: (seq_q, d_k)
    K: (seq_k, d_k)
    V: (seq_k, d_v)
    返回: (seq_q, d_v)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (seq_q, seq_k)
    # softmax（数值稳定版）
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores)
    weights /= weights.sum(axis=-1, keepdims=True)  # (seq_q, seq_k)
    return weights @ V                         # (seq_q, d_v)

# 验证形状
seq_q, seq_k, d_k, d_v = 4, 6, 8, 16
np.random.seed(0)
Q = np.random.randn(seq_q, d_k)
K = np.random.randn(seq_k, d_k)
V = np.random.randn(seq_k, d_v)
out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (seq_q, d_v)
print(f'输出 shape: {out.shape}  ✅')
print(f'为什么除以 √d_k：d_k={d_k} → 除以 {d_k**0.5:.2f}，防止点积过大导致 softmax 饱和')

In [ ]:
attention_review = {
    "time_minutes":  None,
    "stuck_at":      None,
    "correct_parts": None,
    "need_practice": None,
}
print(attention_review)

## ✏️ 题 4：综合题——MFCC 流水线（目标：30 分钟内完成）

这道题考察你能否把多个模块串联起来，不只是单点推导。

**题目**

从麦克风采到一段 16kHz 的 PCM 信号 x，到最终输出 13 维 MFCC 系数，
在白板上写出完整流水线（不需要代码，只需公式和 shape）：

1. 分帧（framing）：写出帧数 T 和帧长 L 的计算公式（设 window=25ms，hop=10ms）
2. 加窗（windowing）：用什么窗函数？写出 Hann 窗的表达式
3. FFT：输出的 shape 是什么？复数还是实数？
4. 功率谱（power spectrum）：从 FFT 输出到功率谱，写出公式
5. Mel 滤波：写出 Hz→Mel 的映射公式；滤波器组矩阵 shape 是什么？
6. 取对数：为什么要取 log？
7. DCT-II：写出 DCT-II 的公式；为什么用 DCT 而不是再做一次 FFT？
8. 切片：最终输出哪几个系数？为什么丢掉第 0 个？

先在纸上完成，再运行对答案格。

In [ ]:
# 对答案格：MFCC 流水线各步骤的 shape 与公式
import numpy as np

sr = 16000
duration = 1.0
x = np.random.randn(int(sr * duration))   # 模拟 1 秒 PCM

# 1. 分帧参数
win_len = int(0.025 * sr)   # 25ms → 400 samples
hop     = int(0.010 * sr)   # 10ms → 160 samples
n_frames = 1 + (len(x) - win_len) // hop
print(f"1. 分帧: {n_frames} 帧 × {win_len} 点/帧")

# 2. Hann 窗
hann = 0.5 * (1 - np.cos(2 * np.pi * np.arange(win_len) / (win_len - 1)))
print(f"2. Hann 窗 shape: {hann.shape}")

# 3. FFT（取前半段）
n_fft = win_len
n_freq = n_fft // 2 + 1
print(f"3. FFT 输出: {n_freq} 个复数频点（取前 {n_freq} 个）")

# 4. 功率谱
frame0 = x[:win_len] * hann
# 注意：aurora.audio.fft 实现了等价的 FFT；此处借用 np.fft.rfft 仅为演示 shape，
# 生产代码请使用 aurora 自有实现，以符合「不使用 API 黑盒」原则。
power = np.abs(np.fft.rfft(frame0, n=n_fft)) ** 2
print(f"4. 功率谱 shape: {power.shape}")

# 5. Mel 滤波
n_mels = 40
def hz_to_mel(f): return 2595 * np.log10(1 + f / 700)
def mel_to_hz(m): return 700 * (10 ** (m / 2595) - 1)
print(f"5. Mel 滤波器组 shape: ({n_mels}, {n_freq})")
print(f"   Hz→Mel: 2595 × log10(1 + f/700)")

# 6. 对数
print("6. log(energy)：压缩动态范围，使低能量区域也可辨别（人耳响度感知近似对数）")

# 7. DCT-II 实现（手写，正交归一化 norm='ortho'，与 L49 / aurora.audio.mfcc.dct_ii 同约定）
def dct2(x):
    """DCT-II（正交归一化）：X[k] = s_k · Σ_{n=0}^{M-1} x[n] · cos(π·k·(2n+1) / (2M))
    s_0 = √(1/M)，s_k = √(2/M) (k>0)——正交矩阵，逆变换即转置，能量守恒。
    """
    M = len(x)
    ns = np.arange(M)
    scale = np.full(M, np.sqrt(2.0 / M))
    scale[0] = np.sqrt(1.0 / M)
    return np.array([
        scale[k] * np.sum(x * np.cos(np.pi * k * (2 * ns + 1) / (2 * M)))
        for k in range(M)
    ])

# 验证 DCT-II 对一个假 Mel 能量向量（对齐 scipy 的 norm='ortho'）
fake_mel_log = np.log(np.abs(np.random.randn(n_mels)) + 1e-8)
cepstrum = dct2(fake_mel_log)
try:
    from scipy.fft import dct as scipy_dct
    ref = scipy_dct(fake_mel_log, type=2, norm='ortho')
    assert np.allclose(cepstrum, ref, atol=1e-9), "DCT-II 与 scipy 参考不一致"
    print(f"7. DCT-II 验证通过（vs scipy.fft.dct, norm='ortho'）✅  输出 shape: {cepstrum.shape}")
except ImportError:
    print(f"7. DCT-II 实现完成（scipy 未安装，跳过交叉验证）shape: {cepstrum.shape}")

# 8. 切片：保留系数 1~13，丢掉第 0 个（均值能量，与响度相关，不含音色信息）
n_mfcc = 13
mfcc_frame = cepstrum[1:n_mfcc + 1]
print(f"8. MFCC 切片后 shape/帧: {mfcc_frame.shape}  (系数 1~{n_mfcc})")

## ✏️ 总体自评：三道题完成后

三道推导题都完成后，做一次完整的自评，
找出你在哪个模块还需要加强，针对性复习，让基础真正扎实、经得起追问。

In [ ]:
# 综合自评（基于前三题 + 综合题的推导情况）
overall_review = {
    # 对每个模块打分：'熟练' / '基本掌握' / '需要加强'
    "FFT 蝶形递推":           None,
    "CTC 前向递推":           None,
    "Attention 矩阵推导":     None,
    "MFCC 全流水线":          None,

    # 耗时最长的一道题
    "耗时最长":               None,   # 例: "CTC"

    # 面试前最需要重新推导一遍的模块
    "优先复习":               None,

    # 下次推导目标（例: "FFT 从 25 分钟压缩到 15 分钟"）
    "下次目标":               None,
}

filled = sum(1 for v in overall_review.values() if v is not None)
熟练 = sum(1 for k in ["FFT 蝶形递推","CTC 前向递推","Attention 矩阵推导","MFCC 全流水线"]
           if overall_review.get(k) == "熟练")

print(f"已填 {filled}/{len(overall_review)} 项")
print(f"熟练模块：{熟练}/4")
if overall_review["优先复习"]:
    print(f"优先复习：{overall_review['优先复习']}")
if overall_review["下次目标"]:
    print(f"下次目标：{overall_review['下次目标']}")

## 本课收束

白板推导是检验你**真懂**还是**背过**最直接的方式。如果某一题超时或卡住：

1. 回到对应课程（FFT → L38-L39，CTC → L68-L69，Attention → L83）
2. 合上 notebook，再推一遍
3. 重复直到 20 分钟内能从头写完

**下一课 L97**：技术沟通——如何在 30 分钟内把 Aurora 任何一个模块讲清楚给别人听（面试只是其中一个场合）。

---

→ **下一课**　[L97 · 面试准备与技术沟通](L97_interview.ipynb)

> 下节课将学习 **面试准备与技术沟通**：技术面试的「黄金结构」（问题→方法→实现→结果→局限）、为 Aurora 每个核心模块准备 30 秒 / 3 分钟两种讲法、如何回答「为什么从零实现、不用 librosa」。